## Introduction to deep learning - exercise

## Example

*Human Activity Recognition using Smartphones* dataset

Dataset description:

*The experiments have been carried out with a group of 30 volunteers. Each person performed six activities
(WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING) wearing a smartphone.
Using its embedded accelerometer and gyroscope, we captured 3-axial linear acceleration and 3-axial angular velocity.
The experiments have been video-recorded to label the data manually.*

**Variables:**
For each record in the dataset it is provided:
* A 561-feature vector with time and frequency domain variables.
* Its activity label.
* An identifier of the subject who carried out the experiment.

More details at: https://archive.ics.uci.edu/ml/datasets/human+activity+recognition+using+smartphones

### Loading and preparing the data

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
folder = ''  ## put here folder where the file HAR_clean.csv is located

Load the dataset: "HAR_clean.csv"

In [3]:
all_data = pd.read_csv(os.path.join(folder, 'HAR_clean.csv'), index_col=0)

Divide into input and output data

In [4]:
input_data = all_data.iloc[:,:-2].values.astype(np.float32)
input_data.shape

(10299, 561)

In [5]:
output_data = all_data.iloc[:,-1].values
output_data.shape

(10299,)

Divide the data into train and test, keeping 30% for the test

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(input_data, output_data, test_size=0.3)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

(7209, 561) (7209,)
(3090, 561) (3090,)


### Best shallow ML model

In [8]:
from sklearn import svm
from sklearn.model_selection import GridSearchCV

parameters = {'kernel':['linear', 'rbf'], 'C':[1, 10, 100,1000], 'gamma':[0.01, 0.001]}

svm_model_d = svm.SVC()
opt_model_d = GridSearchCV(svm_model_d, parameters)

opt_model_d.fit(X_train, y_train)
print (opt_model_d.best_estimator_)

SVC(C=1000, gamma=0.001)


In [9]:
opt_model_d.score(X_test, y_test)

0.9893203883495145

### Training a DL model

In [28]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()

le.fit(y_train)

y_train_encoded = le.transform(y_train)
y_test_encoded = le.transform(y_test)

In [29]:
X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train_encoded)

X_test = torch.tensor(X_test)
y_test = torch.tensor(y_test_encoded)

C:\Users\bruno\AppData\Local\Temp\ipykernel_22328\1811169814.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train)
C:\Users\bruno\AppData\Local\Temp\ipykernel_22328\1811169814.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test)


In [ ]:
# DNN model - define here the model - complete

class HARNet(nn.Module):
    def __init__(self, in_features, n_classes):
        super().__init__()

        self.net = nn
        (
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
            nn.ReLU(),
            nn.Linear(n_classes, n_classes)
        )

    def forward(self, x):
        return self.net(x)


n_features = X_train.shape[1]
n_classes = len(np.unique(y_train))

model = HARNet(n_features, n_classes)

In [35]:
# Loss & optimizer - complete !
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [36]:
from sklearn.metrics import accuracy_score

# Training loop
epochs = 100
batch_size = 128
train_losses = []
test_accuracies = []

for epoch in range(epochs):

    # ---- mini-batch training ----
    perm = torch.randperm(X_train.size(0))
    epoch_loss = 0

    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = X_train[idx], y_train[idx]

        preds = model(xb)
        loss = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    train_losses.append(epoch_loss)

    # ---- evaluation ----
    with torch.no_grad():
        logits = model(X_test)
        preds = torch.argmax(logits, dim=1)
        acc = accuracy_score(y_test.numpy(), preds.numpy())
        test_accuracies.append(acc)

    print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss:.3f} | Test Acc: {acc:.4f}")

Epoch 01 | Loss: 63.237 | Test Acc: 0.5783
Epoch 02 | Loss: 36.380 | Test Acc: 0.8068
Epoch 03 | Loss: 15.515 | Test Acc: 0.9469
Epoch 04 | Loss: 6.536 | Test Acc: 0.9612
Epoch 05 | Loss: 5.092 | Test Acc: 0.9712
Epoch 06 | Loss: 3.861 | Test Acc: 0.9715
Epoch 07 | Loss: 3.327 | Test Acc: 0.9715
Epoch 08 | Loss: 3.127 | Test Acc: 0.9722
Epoch 09 | Loss: 2.745 | Test Acc: 0.9744
Epoch 10 | Loss: 2.694 | Test Acc: 0.9751
Epoch 11 | Loss: 2.307 | Test Acc: 0.9761
Epoch 12 | Loss: 2.244 | Test Acc: 0.9761
Epoch 13 | Loss: 2.223 | Test Acc: 0.9790
Epoch 14 | Loss: 1.849 | Test Acc: 0.9741
Epoch 15 | Loss: 1.805 | Test Acc: 0.9777
Epoch 16 | Loss: 1.788 | Test Acc: 0.9767
Epoch 17 | Loss: 1.624 | Test Acc: 0.9793
Epoch 18 | Loss: 1.286 | Test Acc: 0.9790
Epoch 19 | Loss: 1.407 | Test Acc: 0.9770
Epoch 20 | Loss: 1.258 | Test Acc: 0.9799
Epoch 21 | Loss: 1.251 | Test Acc: 0.9809
Epoch 22 | Loss: 1.659 | Test Acc: 0.9702
Epoch 23 | Loss: 1.228 | Test Acc: 0.9825
Epoch 24 | Loss: 0.873 | Test A

In [37]:
# Test set evaluation
print("\nFinal test accuracy:", test_accuracies[-1])


Final test accuracy: 0.987378640776699
